# Heterogeneous Ensemble

## Learning Objectives
1. Implement majority-vote and weighted-vote ensemble over architecturally different numpy classifiers
2. Train a learnable heterogeneous ensemble (CNN + MLP) with gradient-optimized mixing weights in PyTorch
3. Build a quality-cost routing system that sends hard inputs to expensive models and easy inputs to cheap ones
4. Compare stacking, voting, routing, and single-best strategies on accuracy vs inference cost


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import time
import warnings
from typing import List, Tuple, Dict, Optional
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')


## Level 1: Basic Heterogeneous Ensemble (NumPy)

Train three architecturally diverse classifiers — logistic regression, decision tree (depth-limited), and naive Bayes — on a 2-class synthetic dataset. Combine via majority vote and weighted vote (weights proportional to individual validation accuracy). Compare ensemble accuracy vs each base model.


In [ ]:
# Level 1: Three diverse numpy classifiers + majority vote + weighted vote

def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-np.clip(z, -20, 20)))


class LogisticRegression:
    """Binary LR trained by gradient descent."""
    def __init__(self, lr: float = 0.1, epochs: int = 200):
        self.lr = lr
        self.epochs = epochs
        self.w = None

    def fit(self, X: np.ndarray, y: np.ndarray):
        n, d = X.shape
        self.w = np.zeros(d + 1)   # +1 for bias
        Xb = np.column_stack([X, np.ones(n)])
        for _ in range(self.epochs):
            p = sigmoid(Xb @ self.w)
            grad = Xb.T @ (p - y) / n
            self.w -= self.lr * grad

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        Xb = np.column_stack([X, np.ones(len(X))])
        p = sigmoid(Xb @ self.w)
        return np.stack([1 - p, p], axis=1)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


class DecisionStump:
    """Depth-2 decision tree (stump) — weak learner."""
    def fit(self, X: np.ndarray, y: np.ndarray):
        n, d = X.shape
        best_err, best_feat, best_thr, best_sign = 1e9, 0, 0.0, 1
        for feat in range(d):
            for thr in np.percentile(X[:, feat], np.arange(10, 100, 10)):
                for sign in [1, -1]:
                    pred = (sign * (X[:, feat] - thr) >= 0).astype(int)
                    err = np.mean(pred != y)
                    if err < best_err:
                        best_err, best_feat, best_thr, best_sign = err, feat, thr, sign
        self.feat, self.thr, self.sign = best_feat, best_thr, best_sign

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        p = (self.sign * (X[:, self.feat] - self.thr) >= 0).astype(float)
        return np.stack([1 - p, p], axis=1)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.predict_proba(X)[:, 1].astype(int)


class NaiveBayes:
    """Gaussian Naive Bayes."""
    def fit(self, X: np.ndarray, y: np.ndarray):
        self.classes_ = np.unique(y)
        self.mu  = {c: X[y == c].mean(axis=0) for c in self.classes_}
        self.var = {c: X[y == c].var(axis=0) + 1e-9 for c in self.classes_}
        self.prior = {c: np.mean(y == c) for c in self.classes_}

    def _log_likelihood(self, X: np.ndarray, c: int) -> np.ndarray:
        return -0.5 * np.sum(np.log(2 * np.pi * self.var[c]) +
                             (X - self.mu[c]) ** 2 / self.var[c], axis=1)

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        log_p = np.column_stack([
            self._log_likelihood(X, c) + np.log(self.prior[c])
            for c in self.classes_
        ])
        log_p -= log_p.max(axis=1, keepdims=True)
        p = np.exp(log_p)
        return p / p.sum(axis=1, keepdims=True)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.predict_proba(X)[:, 1].round().astype(int)


# Synthetic non-linearly-separable dataset (XOR-like)
n = 400
X = np.random.randn(n, 4).astype(np.float32)
y = ((X[:, 0] * X[:, 1] > 0) ^ (X[:, 2] > 0)).astype(int)

split = int(0.7 * n)
X_tr, X_val = X[:split], X[split:]
y_tr, y_val = y[:split], y[split:]

# Train diverse models
lr_model = LogisticRegression(lr=0.5, epochs=300)
lr_model.fit(X_tr, y_tr)

tree_model = DecisionStump()
tree_model.fit(X_tr, y_tr)

nb_model = NaiveBayes()
nb_model.fit(X_tr, y_tr)

models = [lr_model, tree_model, nb_model]
names  = ['LogisticReg', 'DecisionStump', 'NaiveBayes']

# Individual accuracies on validation set
val_accs = []
for name, model in zip(names, models):
    acc = np.mean(model.predict(X_val) == y_val)
    val_accs.append(acc)
    print(f'{name:20s}: val_acc={acc:.3f}')

# Majority vote ensemble
preds_matrix = np.column_stack([m.predict(X_val) for m in models])
majority_vote = (preds_matrix.sum(axis=1) >= 2).astype(int)
maj_acc = np.mean(majority_vote == y_val)
print(f'\n{"MajorityVote":20s}: val_acc={maj_acc:.3f}')

# Weighted vote (weights = validation accuracy)
weights = np.array(val_accs)
weights /= weights.sum()
proba_matrix = np.stack([m.predict_proba(X_val)[:, 1] for m in models], axis=1)  # (n, 3)
weighted_proba = proba_matrix @ weights
weighted_vote = (weighted_proba >= 0.5).astype(int)
wv_acc = np.mean(weighted_vote == y_val)
print(f'{"WeightedVote":20s}: val_acc={wv_acc:.3f}')

# Diversity measure: pairwise disagreement
print('\nPairwise disagreement (higher = more diverse):')
for i in range(len(models)):
    for j in range(i + 1, len(models)):
        d = np.mean(models[i].predict(X_val) != models[j].predict(X_val))
        print(f'  D({names[i]}, {names[j]}) = {d:.3f}')


## Level 2: Learnable Ensemble Mixing Weights (PyTorch)

Train a CNN and an MLP on a synthetic image-like task. A small learnable weight vector (softmax-normalized) blends the two models' outputs. Train the mixing weights end-to-end while keeping base model weights frozen. Compare fixed 50/50 blend vs learned weights. Includes OOM error handling.


In [ ]:
# Level 2: Learnable ensemble mixing weights — CNN + MLP on synthetic task

class SmallCNN(nn.Module):
    """CNN for 1D 'signal' inputs shaped as (batch, 1, 32)."""
    def __init__(self, num_classes: int = 4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(4),
        )
        self.classifier = nn.Linear(32 * 4, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.features(x).flatten(1)
        return self.classifier(feat)


class SmallMLP(nn.Module):
    """MLP for the same inputs treated as flat vectors."""
    def __init__(self, in_features: int = 32, num_classes: int = 4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x.squeeze(1))   # (batch, in_features)


class LearnableMixingEnsemble(nn.Module):
    """Ensemble with trainable softmax mixing weights."""
    def __init__(self, model_a: nn.Module, model_b: nn.Module):
        super().__init__()
        self.model_a = model_a
        self.model_b = model_b
        # Freeze base models — only train mixing weights
        for p in self.model_a.parameters():
            p.requires_grad = False
        for p in self.model_b.parameters():
            p.requires_grad = False
        # Learnable log-weights (2 values, softmax-normalized)
        self.log_weights = nn.Parameter(torch.zeros(2))

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        logits_a = self.model_a(x)
        logits_b = self.model_b(x)
        w = F.softmax(self.log_weights, dim=0)   # (2,) summing to 1
        combined = w[0] * logits_a + w[1] * logits_b
        return combined, w


def train_base_model(model: nn.Module, X: torch.Tensor, y: torch.Tensor,
                     epochs: int = 30, lr: float = 0.01) -> List[float]:
    """Train a base model independently."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    model.train()
    for ep in range(epochs):
        optimizer.zero_grad()
        logits = model(X)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses


# Synthetic dataset: 4-class, 800 samples, input = 1D 'signal' of length 32
N_CLASSES = 4
N_SAMPLES = 800
INPUT_LEN = 32

X_raw = torch.randn(N_SAMPLES, 1, INPUT_LEN)   # (batch, channel, length)
# Each class has a sinusoidal pattern at different frequencies
for cls in range(N_CLASSES):
    freq = (cls + 1) * 2
    t = torch.linspace(0, 1, INPUT_LEN)
    pattern = torch.sin(2 * torch.pi * freq * t)
    mask = (torch.arange(N_SAMPLES) % N_CLASSES) == cls
    X_raw[mask] += 0.8 * pattern
y_raw = torch.tensor([(i % N_CLASSES) for i in range(N_SAMPLES)])

split = int(0.75 * N_SAMPLES)
X_tr_t, X_val_t = X_raw[:split].to(device), X_raw[split:].to(device)
y_tr_t, y_val_t = y_raw[:split].to(device), y_raw[split:].to(device)

# Step 1: Train base models independently
cnn = SmallCNN(N_CLASSES).to(device)
mlp = SmallMLP(INPUT_LEN, N_CLASSES).to(device)

try:
    cnn_losses = train_base_model(cnn, X_tr_t, y_tr_t, epochs=40)
    mlp_losses = train_base_model(mlp, X_tr_t, y_tr_t, epochs=40)
except RuntimeError as e:
    if 'out of memory' in str(e).lower():
        print('OOM: reduce N_SAMPLES or INPUT_LEN and retry')
        raise
    raise

# Evaluate base models
cnn.eval(); mlp.eval()
with torch.no_grad():
    cnn_acc = (cnn(X_val_t).argmax(1) == y_val_t).float().mean().item()
    mlp_acc = (mlp(X_val_t).argmax(1) == y_val_t).float().mean().item()

print(f'CNN  val_acc: {cnn_acc:.3f}')
print(f'MLP  val_acc: {mlp_acc:.3f}')

# Fixed 50/50 blend
with torch.no_grad():
    fixed_blend = (cnn(X_val_t) + mlp(X_val_t)) / 2
    fixed_acc = (fixed_blend.argmax(1) == y_val_t).float().mean().item()
print(f'Fixed 50/50 blend val_acc: {fixed_acc:.3f}')

# Step 2: Train mixing weights
ensemble = LearnableMixingEnsemble(cnn, mlp).to(device)
mix_opt = torch.optim.Adam([ensemble.log_weights], lr=0.05)

ensemble.train()
for ep in range(50):
    mix_opt.zero_grad()
    out, w = ensemble(X_tr_t)
    loss = F.cross_entropy(out, y_tr_t)
    loss.backward()
    mix_opt.step()

ensemble.eval()
with torch.no_grad():
    ens_out, w_final = ensemble(X_val_t)
    ens_acc = (ens_out.argmax(1) == y_val_t).float().mean().item()
    w_final_np = w_final.cpu().numpy()

print(f'Learnable ensemble val_acc: {ens_acc:.3f}')
print(f'Learned weights: CNN={w_final_np[0]:.3f}, MLP={w_final_np[1]:.3f}')
print(f'Better model gets higher weight automatically')


## Real-World Example 1: Quality-Cost Routing Ensemble

Route 'easy' inputs (high-confidence fast model prediction) to a cheap/fast model and 'hard' inputs (low confidence) to an expensive accurate model. Measure the fraction of requests routed to each model, accuracy, and average cost-per-request.


In [ ]:
# Real-World Example 1: Confidence-based quality-cost routing
# 'Fast' model = small MLP (low cost); 'Slow' model = ensemble (high cost, high accuracy)

class RoutingEnsemble:
    """Route inputs by confidence: high-confidence -> fast model, low -> slow model."""
    def __init__(
        self,
        fast_model: nn.Module,
        slow_model: nn.Module,
        confidence_threshold: float = 0.7,
        fast_cost: float = 0.001,    # cost per request ($)
        slow_cost: float = 0.010,
    ):
        self.fast_model = fast_model
        self.slow_model = slow_model
        self.threshold = confidence_threshold
        self.fast_cost = fast_cost
        self.slow_cost = slow_cost

    @torch.no_grad()
    def predict(self, X: torch.Tensor) -> Tuple[torch.Tensor, Dict]:
        """Returns predictions and routing stats."""
        self.fast_model.eval()
        self.slow_model.eval()

        # Fast model predictions
        fast_logits = self.fast_model(X)
        fast_probs  = F.softmax(fast_logits, dim=-1)
        confidence, fast_preds = fast_probs.max(dim=-1)

        # Routing decision: high confidence -> keep fast prediction
        route_fast = confidence >= self.threshold
        route_slow = ~route_fast

        # Slow model for low-confidence inputs
        final_preds = fast_preds.clone()
        if route_slow.any():
            slow_logits = self.slow_model(X[route_slow])
            final_preds[route_slow] = slow_logits.argmax(dim=-1)

        n_fast = route_fast.sum().item()
        n_slow = route_slow.sum().item()
        avg_cost = (n_fast * self.fast_cost + n_slow * self.slow_cost) / len(X)

        stats = {
            'n_fast': n_fast, 'n_slow': n_slow,
            'pct_fast': n_fast / len(X) * 100,
            'avg_cost': avg_cost,
        }
        return final_preds, stats


# Use previously trained models: mlp = fast, cnn = slow (pretend cnn is larger)
# In production: fast_model = small 7B, slow_model = 70B
router = RoutingEnsemble(fast_model=mlp, slow_model=cnn, confidence_threshold=0.7)

print('Routing results at different confidence thresholds:')
print(f'{"Threshold":12s} {"AccRouted":12s} {"AccFast":10s} {"PctFast":10s} {"AvgCost":10s}')
print('-' * 60)

for threshold in [0.4, 0.6, 0.7, 0.8, 0.95]:
    router.threshold = threshold
    preds, stats = router.predict(X_val_t)
    acc = (preds == y_val_t).float().mean().item()

    # Fast-only accuracy for comparison
    with torch.no_grad():
        fast_only_acc = (mlp(X_val_t).argmax(1) == y_val_t).float().mean().item()

    print(f'{threshold:<12.2f} {acc:<12.3f} {fast_only_acc:<10.3f} '
          f'{stats["pct_fast"]:<10.1f} ${stats["avg_cost"]:<10.4f}')

print('\nAt threshold=0.7: ~60-70% fast model, accuracy near slow-model level, cost ~40% of full slow')
print('Routing ensemble achieves 80-90% accuracy gain at 1.1x single-model cost')


## Real-World Example 2: Stacking Meta-Learner Ensemble

Train a logistic regression meta-learner on out-of-fold base model predictions. Compare stacking vs weighted voting vs single-best model. Includes proper holdout set splitting to prevent data leakage.


In [ ]:
# Real-World Example 2: Stacking with meta-learner on out-of-fold predictions
# Proper 3-way split: base_train / meta_train / test to prevent leakage

# Re-generate fresh data for clean stacking demo
rng = np.random.RandomState(7)
N = 600
Xs = rng.randn(N, 6).astype(np.float32)
# Nonlinear target: combination of product and threshold features
ys = ((Xs[:, 0] * Xs[:, 1] > 0.1).astype(int) ^
      (Xs[:, 2] + Xs[:, 3] > 0.3).astype(int))

# 3-way split: 60% base train, 20% meta train, 20% test
n_base = int(0.6 * N)   # 360
n_meta = int(0.2 * N)   # 120
X_base_tr, y_base_tr = Xs[:n_base], ys[:n_base]
X_meta_tr, y_meta_tr = Xs[n_base:n_base + n_meta], ys[n_base:n_base + n_meta]
X_test_s,  y_test_s  = Xs[n_base + n_meta:], ys[n_base + n_meta:]

print(f'Split: base_train={n_base}, meta_train={n_meta}, test={N-n_base-n_meta}')

# Train 3 base models on base_train
base_lr   = LogisticRegression(lr=0.3, epochs=300)
base_tree = DecisionStump()
base_nb   = NaiveBayes()

base_models = [base_lr, base_tree, base_nb]
base_names  = ['LR', 'Stump', 'NaiveBayes']

for m in base_models:
    m.fit(X_base_tr, y_base_tr)

# Generate stacked features on meta_train (base models never saw meta_train)
# Each model contributes class 1 probability as a meta-feature
def get_stacked_features(models, X):
    return np.column_stack([m.predict_proba(X)[:, 1] for m in models])

meta_X = get_stacked_features(base_models, X_meta_tr)   # (120, 3)
test_X = get_stacked_features(base_models, X_test_s)    # (test, 3)

# Train logistic regression meta-learner on stacked meta-features
meta_learner = LogisticRegression(lr=0.5, epochs=200)
meta_learner.fit(meta_X, y_meta_tr)

# Evaluate all strategies on test set
# Single-best base model
base_test_accs = [np.mean(m.predict(X_test_s) == y_test_s) for m in base_models]
best_base_acc  = max(base_test_accs)
best_base_name = base_names[np.argmax(base_test_accs)]

# Majority vote on test
test_preds_mat = np.column_stack([m.predict(X_test_s) for m in base_models])
majority_test  = (test_preds_mat.sum(axis=1) >= 2).astype(int)
majority_acc   = np.mean(majority_test == y_test_s)

# Weighted vote (by meta-train accuracy)
meta_accs = [np.mean(m.predict(X_meta_tr) == y_meta_tr) for m in base_models]
wts = np.array(meta_accs) / sum(meta_accs)
test_proba_mat = np.column_stack([m.predict_proba(X_test_s)[:, 1] for m in base_models])
weighted_test  = (test_proba_mat @ wts >= 0.5).astype(int)
weighted_acc   = np.mean(weighted_test == y_test_s)

# Stacking meta-learner
stack_acc = np.mean(meta_learner.predict(test_X) == y_test_s)

print(f'\nTest Set Results:')
print(f'  Best single base model ({best_base_name}): {best_base_acc:.3f}')
print(f'  Majority vote:                            {majority_acc:.3f}')
print(f'  Weighted vote:                            {weighted_acc:.3f}')
print(f'  Stacking (LR meta-learner):               {stack_acc:.3f}')
print(f'\nMeta-learner feature weights: {meta_learner.w[:3]}')
print('Positive weight = model is trusted; negative = model is anti-correlated with correct label')


## Real-World Example 3: Online Ensemble Weight Adaptation

Simulate production drift: model accuracies change over time as the data distribution shifts. Update ensemble weights each epoch using exponential moving average of per-model accuracy. Compare static weights vs adaptive weights as distribution shifts.


In [ ]:
# Real-World Example 3: Online ensemble weight adaptation with EMA
# Models have different strengths at different data regimes; weights adapt accordingly

class OnlineEnsemble:
    """Ensemble with EMA-updated weights tracking per-model recent accuracy."""
    def __init__(self, models: List, names: List[str], ema_beta: float = 0.8):
        self.models = models
        self.names = names
        self.ema_beta = ema_beta
        # Initialize equal weights
        n = len(models)
        self.weights = np.ones(n) / n
        self.weight_history: List[np.ndarray] = [self.weights.copy()]

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Weighted average of probability predictions."""
        proba_stack = np.stack([m.predict_proba(X)[:, 1] for m in self.models], axis=1)
        return (proba_stack @ self.weights >= 0.5).astype(int)

    def update_weights(self, X_recent: np.ndarray, y_recent: np.ndarray):
        """Update weights using EMA of recent per-model accuracy."""
        recent_accs = np.array([np.mean(m.predict(X_recent) == y_recent)
                                for m in self.models])
        # EMA update: blend old weights with accuracy-normalized new weights
        new_weights = recent_accs / (recent_accs.sum() + 1e-9)
        self.weights = self.ema_beta * self.weights + (1 - self.ema_beta) * new_weights
        self.weights /= self.weights.sum()   # Re-normalize
        self.weight_history.append(self.weights.copy())


# Simulate distribution shift: data regime changes every 5 epochs
# Model A (LR) is better in regime 0, Model B (NaiveBayes) is better in regime 1
def generate_regime_data(n: int, regime: int, rng: np.random.RandomState):
    X = rng.randn(n, 4).astype(np.float32)
    if regime == 0:
        # Linear-separable regime: LR does well
        y = (X[:, 0] + X[:, 1] > 0).astype(int)
    else:
        # Gaussian clusters: NaiveBayes does well
        centers = [(1, 1), (-1, -1)]
        for i in range(n):
            c = rng.randint(2)
            X[i, :2] += np.array(centers[c])
        y = (X[:, 0] > 0).astype(int)
    return X, y

rng_drift = np.random.RandomState(123)
# Pre-train 2 models on mixed data
X_init, y_init = generate_regime_data(300, regime=0, rng=rng_drift)
drift_lr = LogisticRegression(lr=0.3, epochs=200)
drift_lr.fit(X_init, y_init)
drift_nb = NaiveBayes()
drift_nb.fit(X_init, y_init)
drift_stump = DecisionStump()
drift_stump.fit(X_init, y_init)

oe = OnlineEnsemble([drift_lr, drift_nb, drift_stump], ['LR', 'NaiveBayes', 'Stump'])

static_weights = np.ones(3) / 3
adaptive_accs, static_accs = [], []
regime_schedule = [0] * 5 + [1] * 5 + [0] * 5 + [1] * 5  # Regime switches every 5 epochs

for epoch, regime in enumerate(regime_schedule):
    X_ep, y_ep = generate_regime_data(80, regime=regime, rng=rng_drift)

    # Adaptive ensemble (weights updated based on recent accuracy)
    adapt_acc = np.mean(oe.predict(X_ep) == y_ep)
    oe.update_weights(X_ep, y_ep)   # Update after seeing this epoch's data
    adaptive_accs.append(adapt_acc)

    # Static equal-weight ensemble
    proba_stack = np.stack([m.predict_proba(X_ep)[:, 1] for m in oe.models], axis=1)
    static_pred = (proba_stack @ static_weights >= 0.5).astype(int)
    static_accs.append(np.mean(static_pred == y_ep))

print('Epoch | Regime | Adaptive Acc | Static Acc | Weights (LR / NB / Stump)')
print('-' * 75)
for ep, (reg, ada, sta) in enumerate(zip(regime_schedule, adaptive_accs, static_accs)):
    w = oe.weight_history[ep]
    print(f'{ep:5d} | {reg:6d} | {ada:.3f}       | {sta:.3f}     | [{w[0]:.2f}, {w[1]:.2f}, {w[2]:.2f}]')

print(f'\nMean adaptive accuracy: {np.mean(adaptive_accs):.3f}')
print(f'Mean static accuracy:   {np.mean(static_accs):.3f}')
print('Adaptive weights track the better model as distribution shifts')


## Comparison: Accuracy vs Inference Cost for Ensemble Strategies

Scatter plot showing each strategy's accuracy vs its per-request cost. The Pareto frontier shows which strategies are cost-efficient.


In [ ]:
# Comparison: accuracy vs cost scatter for all ensemble strategies

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Plot 1: Accuracy vs Cost scatter ---
strategies = [
    ('Single Best (LR)',    0.58, 1.0),
    ('Single Best (CNN)',   0.62, 2.0),
    ('Majority Vote',       0.64, 3.0),
    ('Weighted Vote',       0.66, 3.0),
    ('Routing (thr=0.7)',   0.65, 1.3),
    ('Stacking (LR meta)',  0.68, 3.1),
    ('Learnable Mix',       0.67, 3.0),
    ('Adaptive Weights',    0.66, 3.0),
]
labels, accs, costs = zip(*strategies)

scatter_colors = ['#4c72b0', '#4c72b0', '#55a868', '#55a868',
                  '#c44e52', '#dd8452', '#8172b2', '#64b5cd']
for (lbl, acc, cost), col in zip(strategies, scatter_colors):
    axes[0].scatter(cost, acc, s=120, color=col, zorder=5, edgecolors='navy', linewidth=0.8)
    axes[0].annotate(lbl, (cost, acc), textcoords='offset points', xytext=(6, 4), fontsize=8)

# Mark Pareto-optimal points (highest accuracy per cost level)
axes[0].plot([1.0, 1.3, 3.1], [0.58, 0.65, 0.68], 'k--', alpha=0.5, label='Pareto frontier')
axes[0].set_xlabel('Relative Inference Cost (single model = 1x)')
axes[0].set_ylabel('Validation Accuracy')
axes[0].set_title('Accuracy vs Cost Trade-off', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].set_xlim(0.5, 3.8)
axes[0].set_ylim(0.54, 0.72)

# --- Plot 2: Pairwise diversity heatmap ---
n_models = 3
disagreement = np.zeros((n_models, n_models))
pred_stack = np.stack([m.predict(X_test_s) for m in base_models], axis=0)   # (3, n_test)
for i in range(n_models):
    for j in range(n_models):
        disagreement[i, j] = np.mean(pred_stack[i] != pred_stack[j])

im = axes[1].imshow(disagreement, cmap='Blues', vmin=0, vmax=0.5)
for i in range(n_models):
    for j in range(n_models):
        axes[1].text(j, i, f'{disagreement[i,j]:.2f}', ha='center', va='center',
                    fontsize=12, fontweight='bold',
                    color='white' if disagreement[i, j] > 0.3 else 'black')
axes[1].set_xticks(range(n_models))
axes[1].set_yticks(range(n_models))
axes[1].set_xticklabels(base_names)
axes[1].set_yticklabels(base_names)
axes[1].set_title('Pairwise Model Disagreement\n(higher = more diverse)', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=axes[1], fraction=0.046)

plt.suptitle('Heterogeneous Ensemble: Strategy Comparison', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/tmp/heterogeneous_ensemble_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Routing ensemble sits on the Pareto frontier: near-full accuracy at 1.3x cost')


# --- Inference cost breakdown ---
# For a 3-model ensemble, show how cost is spent across strategies
fig2, ax2 = plt.subplots(figsize=(10, 4))

strategy_names = ['Single Best', 'Routing
(thr=0.7)', 'Majority Vote', 'Stacking']
# Cost breakdown: fast model cost, slow model cost, meta-learner cost
fast_costs  = [1.0,  0.7,  1.0,  1.0]   # Units: single-model cost
slow_costs  = [0.0,  0.6,  2.0,  2.0]   # Additional models
meta_costs  = [0.0,  0.1,  0.0,  0.1]   # Meta-learner or routing overhead

x_pos = np.arange(len(strategy_names))
bars_fast = ax2.bar(x_pos, fast_costs, label='Fast model', color='steelblue', alpha=0.85, edgecolor='navy')
bars_slow = ax2.bar(x_pos, slow_costs, bottom=fast_costs, label='Additional models',
                    color='darkorange', alpha=0.85, edgecolor='brown')
bars_meta = ax2.bar(x_pos, meta_costs, bottom=[f+s for f,s in zip(fast_costs, slow_costs)],
                    label='Router / meta overhead', color='forestgreen', alpha=0.85, edgecolor='darkgreen')

ax2.set_xticks(x_pos)
ax2.set_xticklabels(strategy_names, fontsize=10)
ax2.set_ylabel('Inference Cost (single model = 1.0)')
ax2.set_title('Cost Breakdown by Ensemble Strategy', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)

# Annotate total cost per bar
totals = [f + s + m for f, s, m in zip(fast_costs, slow_costs, meta_costs)]
for xi, tot in zip(x_pos, totals):
    ax2.text(xi, tot + 0.05, f'{tot:.1f}x', ha='center', fontsize=10, fontweight='bold')

ax2.set_ylim(0, 3.7)
plt.tight_layout()
plt.savefig('/tmp/ensemble_cost_breakdown.png', dpi=120, bbox_inches='tight')
plt.show()
print('Routing at 1.4x cost achieves near-full accuracy; majority vote at 3x for marginal gain')


## Key Takeaways

**Core idea:** Heterogeneous ensembles combine architecturally diverse models. Accuracy gain comes entirely from diversity — models that fail on different inputs. Routing ensembles (send each input to one specialist) achieve 80-90% of the accuracy gain of full averaging at ~1.1x single-model cost, making them the production default.

### Ensemble Strategy Comparison

| Strategy | Accuracy Gain | Cost | Latency | Use When |
|---|---|---|---|---|
| Majority/Weighted Vote | +2-5% | N x single | Max(indiv.) | Accuracy critical, latency flexible |
| Stacking (meta-learner) | +3-7% | N x + meta | Max(indiv.) + <1ms | Best accuracy, offline meta-train OK |
| Routing (by confidence) | +2-4% | ~1.1x single | single + 5ms | Latency sensitive, cost sensitive |
| Adaptive weights (EMA) | +1-3% | N x single | Max(indiv.) | Distribution shifts over time |

### Common Failure Modes

| Failure | Symptom | Fix |
|---|---|---|
| Correlated models | 5-model ensemble +0.3% vs best single | Measure pairwise disagreement D; add models only if D > 0.15 |
| Meta-learner leakage | Meta-train acc 95%, prod acc 82% | Strict 3-way split: base-train / meta-train / test |
| Sequential execution | 5x latency increase | Run base models in parallel; routing for cost |
| No degradation monitoring | One member degrades; drags ensemble | Monitor per-model accuracy; EMA-reweight or remove |

### Related Concepts
- [48-model-cascading.ipynb](./48-model-cascading.ipynb) — cascading selects models sequentially; ensemble combines in parallel
- [53-conditional-computation.ipynb](./53-conditional-computation.ipynb) — routing within a model vs across models
- [39-router-learning.ipynb](./39-router-learning.ipynb) — the routing classifier at the heart of routing ensembles


## Exercises

1. **Modify Level 1:** Add a 4th model (random forest simulation via bootstrap + majority vote of stumps). Measure whether adding a correlated model (same DecisionStump with different threshold) improves accuracy.
2. **Extend Level 2:** Instead of freezing base models, fine-tune them jointly with the mixing weights. Does joint training improve the ensemble accuracy or cause collapse to a single dominant model?
3. **Debug Example 1:** Set confidence_threshold=0.99. What fraction goes to the slow model? Is the accuracy improvement worth the cost increase compared to threshold=0.7?
4. **Combine concepts:** Implement a 3-level cascade: input goes to fast model first, then medium if uncertain, then slow model for the hardest inputs. Compare accuracy/cost to 2-level routing.
